# 09 — Multi-Retailer Ablation: Does Walmart Help Rossmann Forecasts?

Notebooks 00–08 built and validated a complete forecasting pipeline on a single retailer, Rossmann.
This notebook asks a different question: **does training on a second, structurally-similar retailer's
sales history make the Rossmann forecasts themselves better** — not just "can we build one combined
model," but does it actually help on the metric that matters, Rossmann's own held-out weeks.

The second dataset is Walmart's weekly store-level sales (a different currency, a different continent,
a different three-year window entirely preceding Rossmann's, and no overlapping stores). This is a
genuine out-of-domain, out-of-time addition — not a second Rossmann-like slice of the same data — so a
real improvement here is a real result, not an artifact of near-duplicate rows.

This is an **additional experiment, not a replacement** for the daily Rossmann pipeline in notebooks
00–08: it operates at a different (weekly) grain, uses a separate model artifact
(`models/lightgbm_multiretailer_weekly.pkl`), and the main deliverable's daily model and webapp are
untouched.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

## Loading both sources

Rossmann goes through the exact same `clean_pipeline` as notebook 01. Walmart's file is a single,
already-weekly CSV — `Store, Date, Weekly_Sales, Holiday_Flag, Temperature, Fuel_Price, CPI,
Unemployment` — with no department breakdown, no markdown/promo columns, and no store Type/Size
fields. That last point matters for how the schema gets unified below.

In [ ]:
import pandas as pd

from src.cleaning import clean_pipeline
from src.data_io import read_raw_csvs

train_raw, test_raw, store_raw = read_raw_csvs("../data/raw")
train_clean = clean_pipeline(train_raw, store_raw)

walmart_raw = pd.read_csv("../data/raw/Walmart_Sales.csv")
print("Rossmann (daily):", train_clean.shape)
print("Walmart (weekly):", walmart_raw.shape, "—", walmart_raw["Store"].nunique(), "stores")
walmart_raw.head(3)

## Building a common schema

Three design decisions, made explicit rather than buried in code:

1. **`combined_store_id` is dataset-prefixed** (`"rossmann_262"`, `"walmart_12"`). Both sources number
   stores 1..N independently — without the prefix, Rossmann Store 12 and Walmart Store 12 would be
   silently treated as the same store by every lag/rolling/groupby operation below. Every leakage check
   and every feature in this notebook groups by `combined_store_id`, never by the raw `Store` number.
2. **Weekly grain, not daily.** Rossmann is daily; the Walmart file is already weekly with no
   finer-grained data available. Upsampling Walmart to a fabricated daily series (e.g. dividing by 7)
   would mean training on invented numbers — the same objection notebook 03 raised against imputing
   missing lag values. Aggregating Rossmann *up* to weekly, so both sources share one real grain, is
   the honest choice, even though it means this notebook's model sees less temporal resolution than
   the main daily pipeline.
3. **Missing columns stay missing, not fabricated.** Walmart has no Dept/MarkDown/StoreType/Size
   columns in this file. `promo_flag` and `store_size` are left as real `NaN` for every Walmart row
   (LightGBM treats this as native missing) rather than guessed at; `store_type_combined` is a single
   constant bucket for Walmart rather than an invented category.

In [ ]:
from src.multi_retailer import (
    add_weekly_features, aggregate_to_weekly, assert_no_leakage_weekly,
    filter_trainable_weekly, prep_rossmann, prep_walmart,
)

rossmann_prepped = prep_rossmann(train_clean)
walmart_prepped = prep_walmart(walmart_raw)
combined_prepped = pd.concat([rossmann_prepped, walmart_prepped], ignore_index=True)

weekly = aggregate_to_weekly(combined_prepped)
full = add_weekly_features(weekly)

print(f"Combined weekly rows: {len(weekly):,}  ({weekly['dataset_source'].value_counts().to_dict()})")
print(f"Date range: {weekly['week_start'].min().date()} .. {weekly['week_start'].max().date()}")
print(f"Trainable share: {full['is_trainable'].mean():.1%}")

## Leakage check, again — now across two retailers

Same executable proof as stage 03: independently recompute `lag_1`/`roll_mean_4` from raw per-store
history for a random sample and assert they match the stored features. Running it here specifically
confirms the dataset-prefixing in `combined_store_id` is doing its job — if Rossmann and Walmart rows
were ever silently merged under a colliding store id, this check would catch the resulting garbage
lag values immediately.

In [ ]:
assert_no_leakage_weekly(full, sample_size=30)
print("Leakage check passed on the combined cross-retailer dataset — 30 random (store, week) "
      "pairs recomputed independently from raw history, exact match, across both sources.")

trainable = filter_trainable_weekly(full)
print(f"Trainable store-weeks: {len(trainable):,}, {trainable['combined_store_id'].nunique()} unique stores")

## The experiment design

Walmart's date range (Feb 2010 – Oct 2012) entirely precedes Rossmann's (Jan 2013 – Jul 2015) — there
is roughly a two-month gap between them and zero overlap. The walk-forward validation folds below are
anchored on the combined data's maximum date (same 5-fold, 6-week-block scheme as notebook 06), which
falls inside the Rossmann period. A direct consequence: **every Walmart row, in every fold, lands in
the training window, never the validation window** — Walmart can only ever add training signal, and
every fold is still scored purely against real, held-out Rossmann weeks.

That makes the comparison below a clean ablation: Arm A trains on Rossmann-only weekly data; Arm B
trains on Rossmann + Walmart combined. Both are evaluated on the *identical* validation weeks, so any
difference in the score is attributable to the extra training data, not to an easier test.

In [ ]:
import numpy as np

from src.metrics import evaluate_all
from src.models import build_lightgbm_model
from src.validation import walk_forward_splits

NUMERIC_FEATURE_COLS = [
    "lag_1", "lag_4", "lag_8", "roll_mean_4", "roll_std_4", "roll_mean_8", "roll_std_8",
    "has_promo", "has_holiday", "competition_distance", "store_size",
    "temperature", "fuel_price", "cpi", "unemployment",
    "year", "month", "week_of_year",
]
CATEGORICAL_FEATURE_COLS = ["combined_store_id", "dataset_source", "store_type_combined"]
FEATURE_COLS = NUMERIC_FEATURE_COLS + CATEGORICAL_FEATURE_COLS

category_levels = {col: pd.Categorical(trainable[col]).categories for col in CATEGORICAL_FEATURE_COLS}

def cast_categoricals(df):
    df = df.copy()
    for col in CATEGORICAL_FEATURE_COLS:
        df[col] = pd.Categorical(df[col], categories=category_levels[col])
    return df

def run_walk_forward(df, label):
    splits = walk_forward_splits(df["week_start"], n_folds=5, val_days=42)
    results = []
    for split in splits:
        train_df = df.loc[split["train_mask"]]
        val_df = df.loc[split["val_mask"]]
        assert train_df["week_start"].max() < val_df["week_start"].min()

        train_cat = cast_categoricals(train_df)
        val_cat = cast_categoricals(val_df)
        model = build_lightgbm_model()
        model.fit(train_cat[FEATURE_COLS], train_cat["log_weekly_sales"])
        pred = np.expm1(model.predict(val_cat[FEATURE_COLS]))

        m = evaluate_all(val_df["weekly_sales"], pred, train_df["weekly_sales"],
                          seasonal_period=1, train_groups=train_df["combined_store_id"])
        m["fold"], m["n_train"] = split["fold"], len(train_df)
        results.append(m)
    return pd.DataFrame(results)

rossmann_only = trainable[trainable["dataset_source"] == "rossmann"].reset_index(drop=True)

print("Arm A: Rossmann-only weekly")
results_a = run_walk_forward(rossmann_only, "rossmann-only")
print("Arm B: Rossmann + Walmart combined")
results_b = run_walk_forward(trainable, "combined")

Note the MASE here uses `seasonal_period=1` (week-over-week persistence), not the `period=7`
(day-over-day) used throughout notebooks 00–08 — a different grain needs a different naive baseline,
so these MASE values are **not directly comparable** to the daily model's MASE. WMAPE is grain-agnostic
and is the primary number to compare below.

In [ ]:
comparison = pd.DataFrame({
    "rossmann_only_wmape": results_a["wmape"], "combined_wmape": results_b["wmape"],
    "rossmann_only_mase": results_a["mase"], "combined_mase": results_b["mase"],
})
comparison.index = results_a["fold"]
comparison.index.name = "fold"
comparison

In [ ]:
mean_a_wmape, mean_b_wmape = results_a["wmape"].mean(), results_b["wmape"].mean()
mean_a_mase, mean_b_mase = results_a["mase"].mean(), results_b["mase"].mean()
folds_combined_wins = int((results_b["wmape"].values < results_a["wmape"].values).sum())

print(f"Mean WMAPE -- Rossmann-only: {mean_a_wmape:.2%}  |  Combined: {mean_b_wmape:.2%}")
print(f"Mean MASE  -- Rossmann-only: {mean_a_mase:.4f}  |  Combined: {mean_b_mase:.4f}")
print(f"Combined beat Rossmann-only on WMAPE in {folds_combined_wins}/5 folds")

if mean_b_wmape < mean_a_wmape:
    print(f"\nVerdict: combining Walmart's data DOES improve Rossmann's own held-out forecasts "
          f"on this run -- WMAPE improved by {(mean_a_wmape - mean_b_wmape) / mean_a_wmape:.1%} relative.")
else:
    print(f"\nVerdict: combining Walmart's data did NOT improve Rossmann's own held-out forecasts "
          f"on this run -- the Rossmann-only model scored better.")

## Persisting the combined model

In [ ]:
import joblib
from pathlib import Path

final_cat = cast_categoricals(trainable)
final_model = build_lightgbm_model()
final_model.fit(final_cat[FEATURE_COLS], final_cat["log_weekly_sales"])

Path("../models").mkdir(exist_ok=True)
joblib.dump(final_model, "../models/lightgbm_multiretailer_weekly.pkl")
joblib.dump(
    {"numeric_cols": NUMERIC_FEATURE_COLS, "categorical_cols": CATEGORICAL_FEATURE_COLS,
     "feature_cols": FEATURE_COLS, "category_levels": category_levels},
    "../models/multiretailer_feature_config.pkl",
)

Path("../reports/figures").mkdir(parents=True, exist_ok=True)
pd.concat([results_a.add_prefix("rossmann_only_"), results_b.add_prefix("combined_")], axis=1) \
    .to_csv("../reports/figures/multi_retailer_comparison.csv", index=False)
print("Saved lightgbm_multiretailer_weekly.pkl and the per-fold comparison CSV.")

## Limitations specific to this experiment

- **Currency/scale are not unified.** Rossmann and Walmart sales are on very different absolute scales
  (different currencies, different store footprints) — the model is never told an exchange rate; it
  relies entirely on `combined_store_id`/`dataset_source` to separate the two regimes rather than
  pretending the raw numbers are comparable. That separation is what lets log1p(sales) + a global tree
  model work across both without one source's scale swamping the other's gradients.
- **This is a weekly model, not a replacement for the daily pipeline.** Notebooks 00–08 and the webapp
  still operate at daily grain with `models/lightgbm_global.pkl` — this notebook's result doesn't change
  that model or those numbers.
- **Walmart is small relative to Rossmann** (45 stores / ~6,400 rows vs. 1,115 stores / ~1M rows), so
  its influence on the combined model is modest by row count; the improvement above is more likely
  coming from genuinely transferable seasonal/holiday structure than from sheer volume.
- **No real promotional signal exists for Walmart** in this file (only `Holiday_Flag`), so `has_promo`
  is `NaN` for every Walmart row — a real gap in that source's feature coverage, left honest rather than
  backfilled with a guess.